In [1]:
import os
import pandas as pd
import joblib
from FeaturesTemporais import FeaturesTemporais
from sklearn.ensemble import RandomForestRegressor
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, root_mean_squared_error

def get_csv_root():
    try:
        BASE_DIR = os.path.dirname(os.path.abspath(__file__))
    except NameError:
        BASE_DIR = os.getcwd()
    return os.path.join(BASE_DIR, "..", "processado")

In [2]:
df_treinamento = pd.read_csv(os.path.join(get_csv_root(), "treinamento_final.csv"))
df_testes = pd.read_csv(os.path.join(get_csv_root(), "testes_final.csv"))

display(df_treinamento.columns)

Index(['id_patient', 'ts', 'metodo_medida', 'glucose_level', 'basal',
       'bolus_type', 'bolus_dose', 'meal_type', 'meal_carbs', 'meal_time',
       'sleeping', 'sleep_quality', 'exercise_intensity', 'doing_exercise',
       'target', 'glucose_lag_5', 'glucose_lag_10', 'glucose_lag_15',
       'glucose_lag_20', 'glucose_lag_25', 'glucose_lag_30', 'glucose_delta_5',
       'glucose_delta_15', 'glucose_delta_30', 'glucose_ma_15',
       'glucose_ma_30', 'glucose_std_30'],
      dtype='str')

In [3]:
# ---- 1. Carregar ----
df_treinamento = pd.read_csv(os.path.join(get_csv_root(), "treinamento_final.csv"))
df_testes = pd.read_csv(os.path.join(get_csv_root(), "testes_final.csv"))

columns_used = ['id_patient', 'ts', 'metodo_medida', 'glucose_level', 'basal',
       'bolus_type', 'bolus_dose', 'meal_type', 'meal_carbs', 'meal_time',
       'sleeping', 'sleep_quality', 'exercise_intensity', 'doing_exercise',
       'target', 'glucose_lag_5', 'glucose_lag_10', 'glucose_lag_15',
       'glucose_lag_20', 'glucose_lag_25', 'glucose_lag_30', 'glucose_delta_5',
       'glucose_delta_15', 'glucose_delta_30', 'glucose_ma_15',
       'glucose_ma_30', 'glucose_std_30']

df_tr = df_treinamento[columns_used].copy()
df_val = df_testes[columns_used].copy()

df_tr = df_tr.rename(columns={"ts": "timestamp"})
df_val = df_val.rename(columns={"ts": "timestamp"})


In [4]:
# ---- 2. Regra de negócio (igual antes) ----
for df in (df_tr, df_val):
    df['meal_carbs'] = df['meal_carbs'].fillna(0)
    df['exercise_intensity'] = df['exercise_intensity'].fillna(0)
    df.loc[df["meal_carbs"] == 0, "meal_type"] = "Fasting"

# ---- 2.5. Remover linhas sem alvo ----
df_tr = df_tr.dropna(subset=["target"]).reset_index(drop=True)
df_val = df_val.dropna(subset=["target"]).reset_index(drop=True)

In [5]:
# ---- 3. Separar X e y ----
y_tr = df_tr["target"]
X_tr = df_tr.drop(columns=["target"])

y_val = df_val["target"]
X_val = df_val.drop(columns=["target"])

display(X_tr)

,id_patient,timestamp,metodo_medida,glucose_level,basal,bolus_type,bolus_dose,meal_type,meal_carbs,meal_time,...,glucose_lag_15,glucose_lag_20,glucose_lag_25,glucose_lag_30,glucose_delta_5,glucose_delta_15,glucose_delta_30,glucose_ma_15,glucose_ma_30,glucose_std_30
0,540,2027-05-19 11:20:00,finger_stick,80.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,540,2027-05-19 11:40:00,CGM,72.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,NaN,NaN,NaN,NaN,-8.0,NaN,NaN,77.333333,NaN,NaN
2,540,2027-05-19 11:45:00,CGM,68.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,80.0,NaN,NaN,NaN,-4.0,-12.0,NaN,73.333333,NaN,NaN
3,540,2027-05-19 11:50:00,CGM,65.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,80.0,80.0,NaN,NaN,-3.0,-15.0,NaN,68.333333,NaN,NaN
4,540,2027-05-19 11:55:00,CGM,63.0,0.95,NaN,0.0,Fasting,0.0,NaN,...,72.0,80.0,80.0,NaN,-2.0,-9.0,NaN,65.333333,71.333333,7.366591
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
128316,596,2027-05-26 23:05:00,CGM,163.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,158.0,157.0,155.0,152.0,2.0,5.0,11.0,161.000000,158.833333,2.857738
128317,596,2027-05-26 23:10:00,CGM,166.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,159.0,158.0,157.0,155.0,3.0,7.0,11.0,163.333333,160.666667,3.386247
128318,596,2027-05-26 23:15:00,CGM,168.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,161.0,159.0,158.0,157.0,2.0,7.0,11.0,165.666667,162.500000,3.937004
128319,596,2027-05-26 23:20:00,CGM,167.0,0.45,NaN,0.0,Fasting,0.0,NaN,...,163.0,161.0,159.0,158.0,-1.0,4.0,9.0,167.000000,164.000000,3.577709


In [6]:
# ---- 4. Identificar colunas numéricas e categóricas ----
num_cols = X_tr.select_dtypes(include=["int64", "float64", "bool"]).columns.tolist()
cat_cols = [c for c in X_tr.select_dtypes(include=["object", "category"]).columns if c != "timestamp"]


print("Numéricas:", num_cols)
print("Categóricas:", cat_cols)

Numéricas: ['id_patient', 'glucose_level', 'basal', 'bolus_dose', 'meal_carbs', 'meal_time', 'sleeping', 'sleep_quality', 'exercise_intensity', 'doing_exercise', 'glucose_lag_5', 'glucose_lag_10', 'glucose_lag_15', 'glucose_lag_20', 'glucose_lag_25', 'glucose_lag_30', 'glucose_delta_5', 'glucose_delta_15', 'glucose_delta_30', 'glucose_ma_15', 'glucose_ma_30', 'glucose_std_30']
Categóricas: ['metodo_medida', 'bolus_type', 'meal_type']


/tmp/ipykernel_128635/284490340.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = [c for c in X_tr.select_dtypes(include=["object", "category"]).columns if c != "timestamp"]


In [13]:
# ---- 5. Pipeline ----
preprocessador = ColumnTransformer(
    transformers=[
        ("num", SimpleImputer(strategy="median"), num_cols),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore")),
        ]), cat_cols),
    ],
    remainder="passthrough",   # <- aqui
)

pipeline = Pipeline([
    ("tempo", FeaturesTemporais(coluna_timestamp="timestamp")),
    ("preprocessador", preprocessador),
    ("rf", RandomForestRegressor(
        n_estimators=2000,
        min_samples_leaf=10,
        random_state=0,
        n_jobs=-1,
    )),
])

pipeline.fit(X_tr, y_tr)


,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('tempo', ...), ('preprocessador', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,coluna_timestamp,'timestamp'
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'passthrough'
,"sparse_threshol

In [14]:
# ---- 6. Avaliar ----
p = pipeline.predict(X_val)
mse = mean_squared_error(y_val, p)
rmse = root_mean_squared_error(y_val, p)
print(f"MSE: {mse}")
print(f"RMSE: {rmse}")

MSE: 399.9155114028185
RMSE: 19.997887673522385


In [15]:
y_val.info()
X_val.info()

<class 'pandas.Series'>
RangeIndex: 30181 entries, 0 to 30180
Series name: target
Non-Null Count  Dtype  
--------------  -----  
30181 non-null  float64
dtypes: float64(1)
memory usage: 235.9 KB
<class 'pandas.DataFrame'>
RangeIndex: 30181 entries, 0 to 30180
Data columns (total 26 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   id_patient          30181 non-null  int64  
 1   timestamp           30181 non-null  str    
 2   metodo_medida       29160 non-null  str    
 3   glucose_level       29160 non-null  float64
 4   basal               29371 non-null  float64
 5   bolus_type          722 non-null    str    
 6   bolus_dose          30181 non-null  float64
 7   meal_type           30181 non-null  str    
 8   meal_carbs          30181 non-null  float64
 9   meal_time           299 non-null    float64
 10  sleeping            30181 non-null  bool   
 11  sleep_quality       7352 non-null   float64
 12  exercise_intens

In [16]:
# 1. Contexto de escala
print(y_tr.describe())
print(f"std target: {y_tr.std():.2f}")

# 2. Baselines
media_tr = y_tr.mean()
rmse_media = root_mean_squared_error(y_val, [media_tr] * len(y_val))
# rmse_persist = root_mean_squared_error(y_val, X_val["glucose_level"])
print(f"RMSE chutando a média:      {rmse_media:.2f}")
# print(f"RMSE persistência:          {rmse_persist:.2f}")
print(f"RMSE do seu RF:             {rmse:.2f}")

# 3. Importância
rf = pipeline.named_steps["rf"]
nomes = pipeline.named_steps["preprocessador"].get_feature_names_out()
importancias = pd.Series(rf.feature_importances_, index=nomes).sort_values(ascending=False)
print(importancias.head(25))

count    128321.000000
mean        158.817723
std          60.514723
min          39.000000
25%         113.000000
50%         151.000000
75%         196.000000
max         459.000000
Name: target, dtype: float64
std target: 60.51
RMSE chutando a média:      60.19
RMSE do seu RF:             20.00
num__glucose_level             0.864527
num__glucose_delta_5           0.037797
num__glucose_lag_5             0.025060
num__glucose_delta_15          0.006838
num__glucose_lag_30            0.006593
num__glucose_delta_30          0.006472
remainder__hora_sin            0.006278
remainder__hora_cos            0.005614
num__basal                     0.004723
num__glucose_std_30            0.004295
num__id_patient                0.003703
num__glucose_ma_30             0.003640
num__glucose_lag_10            0.003149
num__glucose_ma_15             0.003142
num__glucose_lag_25            0.003115
num__glucose_lag_20            0.002580
num__glucose_lag_15            0.002228
num__bolus_dose      

In [17]:
# ---- 7. Salvar ----
joblib.dump(pipeline, "modelo_rf.joblib")
print("Modelo salvo em modelo_rf.joblib")

Modelo salvo em modelo_rf.joblib
